In [ ]:
!wget -q -c -O rev1-benchmark-data.tar.gz https://github.com/huboqiang/omnialigner/releases/download/rev1/rev1-benchmark-data.tar.gz
!git clone --quiet --depth 1 --branch rev1 https://github.com/huboqiang/omnialigner.git omnialigner-rev1

from pathlib import Path
import hashlib, tarfile, sys
assert (3, 10) <= sys.version_info[:2] < (3, 12), "Use a Python 3.10 or 3.11 Jupyter kernel."
archive = Path("rev1-benchmark-data.tar.gz").resolve()
checkout = Path("omnialigner-rev1").resolve()
assert archive.is_file() and (checkout / ".git").is_dir(), "Download or clone failed; inspect the messages above."
assert hashlib.sha256(archive.read_bytes()).hexdigest() == "9b98464f0b6625bae48514c601b740caee50aeb008525f103a362fc7a630d7d3", "Data archive checksum mismatch"
destination = checkout / "benchmarks/rev1"
with tarfile.open(archive, "r:gz") as bundle:
    for member in bundle.getmembers():
        if member.name == "NCS" or member.name.startswith("NCS/"):
            member.name = "OmniAligner" + member.name[3:]
        target = (destination / member.name).resolve()
        assert target.is_relative_to(destination) and not (member.issym() or member.islnk()), "Unsafe archive member"
    bundle.extractall(destination)
PACKAGE_ROOT = destination / "OmniAligner"
print("Downloaded and verified rev1 data; package:", PACKAGE_ROOT.name)


In [ ]:
%pip install -q -r {PACKAGE_ROOT / "requirements.txt"}
%cd {PACKAGE_ROOT}


# OmniAligner: calculate metrics and reproduce the submitted comparison

The first part shows every metric function and directly computes all **44 adjacent pairs / 36 OmniAligner means** from released expression inputs and selected coordinates. The second reconstructs the remaining algorithms from their complete pair-score files or archived baseline means, then verifies the **entire submitted table and all 36 ranks**. It does not refit the comparators.

Keep the cells in order. The six metric definitions are visible below; there is no dynamic module loading or opaque scoring wrapper. Allow 48–64 GB RAM for MLN. Failed/undefined/incomplete comparison results retain their original eligibility and support.

The release data contain exact original expression values for the frozen evaluation-gene panel, original cell IDs, annotations and raw coordinates, exported losslessly as sparse NPZs. They are sufficient for the scoring computation; full H5AD downloads are linked separately. Input hashes bind the exported arrays to the audited source H5ADs. No benchmark-server directory is required.

## 1. Load the downloaded benchmark inputs
The paths below are relative to the cloned package. All 18 MLN pairs are included. Each dataset is saved after scoring; reference values never fill a missing or failed calculation.

In [2]:
from pathlib import Path
import os, sys, json, csv, hashlib, importlib.metadata, warnings, time
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
from IPython.display import display

def locate_repository():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "notebooks/OmniAligner_NCS_manifest.json").is_file():
            return candidate
    raise FileNotFoundError("Run from this repository or its notebooks directory")
REPO_ROOT = locate_repository()
DATA_ROOT = REPO_ROOT / "data"
INPUT_ROOT = DATA_ROOT
input_records = json.loads((INPUT_ROOT / "evaluation_inputs_manifest.json").read_text())
input_index = {(x["dataset"], x["layer"]): x for x in input_records}
OUTPUT_ROOT = REPO_ROOT / "results" / "omni"
DATASETS = ["DLPFC", "MHPR", "MMPC", "MPC", "E14_16", "MLN"]
VERIFY_SOURCE_HASHES = True
ATOL = 1e-10
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
METRICS = ["PCC", "CosSim", "SSIM", "MI", "MSE", "Region"]
NATIVE_KEYS = ["PCC", "CosSim", "SSIM", "MI", "Expression_MSE_native_ASSD_bygrid", "Accuracy"]
manifest = json.loads((REPO_ROOT / "notebooks/OmniAligner_NCS_manifest.json").read_text())
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()
def save_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False))
    temporary.replace(path)
reference_path = REPO_ROOT / "notebooks/OmniAligner_NCS_reference.csv"
assert sha256(reference_path) == manifest["expected_table_sha256"]
reference = pd.read_csv(reference_path).set_index("Dataset")
print("Reference values (not recomputed yet)")
display(reference[METRICS].round(6))

Reference values (not recomputed yet)


,PCC,CosSim,SSIM,MI,MSE,Region
Dataset,,,,,,
DLPFC,0.547930,0.783503,0.531970,0.299724,0.127579,0.884759
MHPR,0.540881,0.869558,0.527731,0.252730,1.218373,0.759039
MMPC,0.197483,0.666920,0.193019,0.069390,81.048677,0.871546
MPC,0.333290,0.625865,0.323609,0.152123,0.341752,0.978106
E14_16,0.411842,0.947845,0.414820,0.191671,0.125625,0.643624
MLN,0.812007,0.982588,0.811243,0.809955,0.055572,0.406042


## 2. Verify inputs and the frozen scorer
Hashes bind the exact source H5ADs, selected coordinates, nine checkpoints and original scoring implementation. The six metric directions are PCC/CosSim/SSIM/MI/Region ↑ and MSE ↓. MSE scales differ across datasets; no cross-dataset mean is calculated.

In [3]:
sys.path.insert(0, str(REPO_ROOT / "src"))

inventory = []
for dataset in DATASETS:
    config = manifest["datasets"][dataset]
    assert sha256(DATA_ROOT / config["coordinates"]) == config["coordinates_sha256"], dataset
    for item in config["weights"]:
        assert sha256(DATA_ROOT / item["path"]) == item["sha256"], item["path"]
    for layer in range(len(config["sections"])):
        record = input_index[(dataset, layer)]
        source = INPUT_ROOT / record["path"]
        assert source.is_file(), source
        assert record["source_h5ad_sha256"] == config["source_sha256"][layer]
        if VERIFY_SOURCE_HASHES:
            assert sha256(source) == record["sha256"], source
    inventory.append({"Dataset": dataset, "Sections": len(config["sections"]), "Pairs": config["pairs"],
                      "Original cells": config["original_cells"], "Scored cells": config["evaluation_cells"],
                      "Gene arguments": len(config["evaluation_genes"]), "Unique genes": len(set(config["evaluation_genes"]))})
display(pd.DataFrame(inventory).set_index("Dataset"))
save_json(OUTPUT_ROOT / "INPUT_VALIDATION.json", {"source_hashes_checked": VERIFY_SOURCE_HASHES, "datasets": DATASETS, "inventory": inventory})

,Sections,Pairs,Original cells,Scored cells,Gene arguments,Unique genes
Dataset,,,,,,
DLPFC,4,3,14243,14243,123,117
MHPR,5,4,28317,28317,155,155
MMPC,3,2,3190,3190,100,100
MPC,3,2,5257,5250,79,79
E14_16,16,15,15295,15150,100,100
MLN,19,18,1097769,1097769,100,100


## Metric calculations, shown explicitly

For every adjacent section pair, `coords` contains the two XY arrays and a shared **10 × 10 overlap grid**. For each gene, `X` contains its two expression vectors. We call the functions below directly: `for metric in [PCC, CosSim, SSIM, MI, MSE]: metric(X, coords)`. `Region(labels, coords)` measures nearest-neighbor regional-label agreement.

| Metric | Calculation |
|---|---|
| PCC ↑ | Pearson correlation between the two vectors of grid-mean expression |
| CosSim ↑ | Dot product divided by the product of vector norms |
| SSIM ↑ | Native luminance × contrast × structure, after separately scaling each vector by its maximum |
| MI ↑ | Native `mutual_info_regression`, with seed 0 reset for each gene |
| MSE ↓ | Mean squared difference between the unscaled grid-mean vectors |
| Region ↑ | Fraction of moving-section labels matching the nearest reference label inside the overlap box |

The grid uses the original half-open bin boundaries and native density threshold. Region uses the original closed overlap bounds. Gene arguments (including repeated names) and adjacent pairs receive equal weights. Missing/nonfinite values are not silently dropped. Native insufficient-grid fallback values remain explicit and are excluded from eligible comparison ranks.

In [4]:
import math
from scipy.stats import pearsonr
from scipy.spatial import cKDTree
from sklearn.feature_selection import mutual_info_regression

### Shared spatial support

In [5]:
def shared_grid(reference_xy, moving_xy):
    """Use the original overlap box, half-open bins and native density threshold."""
    xy = (reference_xy, moving_xy)
    x_min = max(np.min(reference_xy[:, 0]), np.min(moving_xy[:, 0]))
    x_max = min(np.max(reference_xy[:, 0]), np.max(moving_xy[:, 0]))
    y_min = max(np.min(reference_xy[:, 1]), np.min(moving_xy[:, 1]))
    y_max = min(np.max(reference_xy[:, 1]), np.max(moving_xy[:, 1]))
    bins = []
    for i in range(10):
        for j in range(10):
            x_start = x_min + i * (x_max - x_min) / 10
            y_start = y_min + j * (y_max - y_min) / 10
            x_end = x_start + (x_max - x_min) / 10
            y_end = y_start + (y_max - y_min) / 10
            bins.append((x_start, x_end, y_start, y_end))

    def inside(points, box):
        x_start, x_end, y_start, y_end = box
        return ((points[:, 0] >= x_start) & (points[:, 0] < x_end)
                & (points[:, 1] >= y_start) & (points[:, 1] < y_end))

    counts = [[np.sum(inside(points, box)) for box in bins] for points in xy]
    threshold = min(np.mean(counts[0]), np.mean(counts[1])) / 2
    labels = [np.full(len(points), None, dtype=object) for points in xy]
    counter = 1
    for index, box in enumerate(bins):
        keep = counts[0][index] >= threshold and counts[1][index] >= threshold
        name = f"Grid_{counter}" if keep else "NonOverlap"
        for points, values in zip(xy, labels):
            values[inside(points, box)] = name
        if keep:
            counter += 1
    return {"xy": xy, "grid_labels": labels}

### Expression means in the retained bins

In [6]:
def grid_expression(X, coords):
    """Mean expression per retained bin, sorted by the native string grid label."""
    means = []
    for expression, labels in zip(X, coords["grid_labels"]):
        frame = pd.DataFrame({"expression": expression, "grid_label": labels})
        frame = frame[(frame.grid_label != "NonOverlap") & frame.grid_label.notnull()]
        means.append(frame.groupby("grid_label").mean().sort_index()["expression"].to_numpy())
    return means

### 1. Pearson correlation

In [7]:
def PCC(X, coords):
    a, b = grid_expression(X, coords)
    return 0.0 if a.size < 2 or b.size < 2 else pearsonr(a, b).statistic

### 2. Cosine similarity

In [8]:
def CosSim(X, coords):
    a, b = grid_expression(X, coords)
    return 0.0 if a.size < 2 or b.size < 2 else a.dot(b) / (np.linalg.norm(a) * np.linalg.norm(b))

### 3. Structural similarity

In [9]:
def SSIM(X, coords):
    a, b = grid_expression(X, coords)
    if a.size < 2 or b.size < 2:
        return 0.0
    # Native convention: separately divide each gene's grid vector by its maximum.
    a, b = a / a.max(), b / b.max()
    mu_a, mu_b = a.mean(), b.mean()
    sigma_a = np.sqrt(((a - mu_a) ** 2).mean())
    sigma_b = np.sqrt(((b - mu_b) ** 2).mean())
    covariance = ((a - mu_a) * (b - mu_b)).mean()
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    C3 = C2 / 2
    luminance = (2 * mu_a * mu_b + C1) / (mu_a ** 2 + mu_b ** 2 + C1)
    contrast = (2 * sigma_a * sigma_b + C2) / (sigma_a ** 2 + sigma_b ** 2 + C2)
    structure = (covariance + C3) / (sigma_a * sigma_b + C3)
    return luminance * contrast * structure

### 4. Mutual information

In [10]:
def MI(X, coords):
    a, b = grid_expression(X, coords)
    if a.size < 2 or b.size < 2:
        return 0.0
    # Reset the seed as in the frozen scorer, including its jitter convention.
    np.random.seed(0)
    return mutual_info_regression(a.reshape(-1, 1), b)[0]

### 5. Expression mean squared error

In [11]:
def MSE(X, coords):
    a, b = grid_expression(X, coords)
    if a.size < 2 or b.size < 2:
        return 100.0  # Native fallback, excluded from eligible table rankings.
    return np.mean(np.square(a - b))

### 6. Regional-label agreement

In [12]:
def Region(labels, coords):
    """Moving-to-reference nearest-neighbor label agreement in the overlap box."""
    reference_xy, moving_xy = coords["xy"]
    x_min = max(np.min(reference_xy[:, 0]), np.min(moving_xy[:, 0]))
    x_max = min(np.max(reference_xy[:, 0]), np.max(moving_xy[:, 0]))
    y_min = max(np.min(reference_xy[:, 1]), np.min(moving_xy[:, 1]))
    y_max = min(np.max(reference_xy[:, 1]), np.max(moving_xy[:, 1]))
    keep = [((points[:, 0] >= x_min) & (points[:, 0] <= x_max)
             & (points[:, 1] >= y_min) & (points[:, 1] <= y_max))
            for points in (reference_xy, moving_xy)]
    reference_xy, moving_xy = reference_xy[keep[0]], moving_xy[keep[1]]
    reference_labels = np.asarray(labels[0])[keep[0]]
    moving_labels = np.asarray(labels[1])[keep[1]]
    _, neighbors = cKDTree(reference_xy).query(moving_xy)
    if len(neighbors) == 0:
        return 0.0
    matches = sum(reference_labels[target] == moving_labels[index]
                  for index, target in enumerate(neighbors)
                  if target < len(reference_labels) and index < len(moving_labels))
    return matches / len(neighbors)

## 3. Join selected coordinates to original cells
The join uses `s<section index>_<original obs_name>`, with an exact one-to-one identity check. For MPC and E14–16 the saved `historical_cohort` mask reproduces the reported common-cell evaluation; the additional original cells are not silently included. DLPFC preserves 123 gene arguments (117 distinct genes), including the historical multiplicity. All other gene orders are also unchanged.

Coordinates remain in the saved benchmark frame, without rescaling or a physical-micrometre claim. Expression is neither normalised nor otherwise altered here.

In [13]:
def load_scoring_slice(dataset, layer, coordinates):
    config = manifest["datasets"][dataset]
    record = input_index[(dataset, layer)]
    with np.load(INPUT_ROOT / record["path"], allow_pickle=False) as raw:
        matrix = sp.csr_matrix((raw["data"], raw["indices"], raw["indptr"]), shape=tuple(raw["shape"]))
        source = ad.AnnData(X=matrix)
        source.obs_names = raw["ids"].astype(str)
        source.var_names = raw["genes"].astype(str)
        source.obs[config["label_column"]] = raw["labels"].astype(str)
        source.obsm["spatial"] = raw["raw_xy"]
    try:
        wanted = [f"s{layer}_{value}" for value in source.obs_names]
        selected = coordinates["layers"] == layer
        ids = coordinates["ids"][selected].tolist()
        assert len(set(ids)) == len(ids) == len(wanted)
        assert set(ids) == set(wanted), (dataset, layer, "Cell identity mismatch")
        lookup = {value: index for index, value in enumerate(ids)}
        order = np.asarray([lookup[value] for value in wanted])
        xy = coordinates["nonrigid"][selected][order]
        keep = coordinates["historical_cohort"][selected][order] if "historical_cohort" in coordinates.files else np.ones(source.n_obs, dtype=bool)
        assert np.isfinite(xy).all() and xy.shape == (source.n_obs, 2)
        unique_genes = list(dict.fromkeys(config["evaluation_genes"]))
        data = source[keep, unique_genes].copy()
        data.obsm["spatial"] = xy[keep]
        data.obs["Region"] = data.obs[config["label_column"]].astype(str)
        if not sp.issparse(data.X):
            data.X = sp.csr_matrix(data.X)
        return data
    finally:
        if source.isbacked:
            source.file.close()

## 4. Recompute every adjacent pair
For each gene, call the five molecular metric functions shown above with `X` and `coords`; then calculate regional-label agreement separately. Average genes within each pair and give each adjacent pair equal weight. Non-finite values propagate rather than being silently dropped.


In [14]:
recomputed_rows = []
started = time.time()
for dataset in DATASETS:
    config = manifest["datasets"][dataset]
    target = OUTPUT_ROOT / dataset
    target.mkdir(exist_ok=True)
    pair_rows, gene_rows = [], []
    with np.load(DATA_ROOT / config["coordinates"], allow_pickle=False) as coordinates:
        assert len(coordinates["ids"]) == config["original_cells"]
        assert set(coordinates["layers"].tolist()) == set(range(len(config["sections"])))
        reference_slice = load_scoring_slice(dataset, 0, coordinates)
        for pair_index in range(config["pairs"]):
            moving_slice = load_scoring_slice(dataset, pair_index + 1, coordinates)
            reference_section, moving_section = reference_slice, moving_slice
            gene_arguments = config["evaluation_genes"]
            coords = shared_grid(reference_section.obsm["spatial"], moving_section.obsm["spatial"])
            gene_records = []
            for gene_name in gene_arguments:
                X = (reference_section.X[:, reference_section.var_names == gene_name].toarray().squeeze(),
                     moving_section.X[:, moving_section.var_names == gene_name].toarray().squeeze())
                gene_result = {"gene": gene_name}
                for metric, key in zip([PCC, CosSim, SSIM, MI, MSE], NATIVE_KEYS[:5]):
                    np.random.seed(0)
                    with warnings.catch_warnings(record=True) as caught:
                        warnings.simplefilter("always")
                        try:
                            value = float(metric(X, coords))
                            gene_result[key] = value if math.isfinite(value) else None
                            gene_result[key + "_error"] = "" if math.isfinite(value) else "native_nonfinite"
                        except Exception as error:
                            gene_result[key] = None
                            gene_result[key + "_error"] = repr(error)
                    gene_result[key + "_warnings"] = " | ".join(str(w.message) for w in caught)
                gene_records.append(gene_result)

            # Equal gene-argument weights. A missing gene score invalidates that pair mean.
            values = {}
            for key in NATIVE_KEYS[:5]:
                numbers = [row[key] for row in gene_records]
                assert len(numbers) == len(gene_arguments)
                values[key] = math.fsum(numbers) / len(numbers) if all(v is not None and math.isfinite(v) for v in numbers) else None

            labels = (reference_section.obs["Region"].to_numpy(), moving_section.obs["Region"].to_numpy())
            values["Accuracy"] = float(Region(labels, coords))
            valid = [pd.notna(g) & (g != "NonOverlap") for g in coords["grid_labels"]]
            valid_grids = [len(set(g[mask])) for g, mask in zip(coords["grid_labels"], valid)]
            support = {"reference_cells": reference_section.n_obs, "moving_cells": moving_section.n_obs,
                       "reference_covered": int(valid[0].sum()), "moving_covered": int(valid[1].sum()),
                       "reference_valid_grids": valid_grids[0], "moving_valid_grids": valid_grids[1],
                       "native_insufficient_grid_fallback": bool(min(valid_grids) < 2)}
            assert [row["gene"] for row in gene_records] == config["evaluation_genes"]
            pair_rows.append({"Dataset": dataset, "pair_index": pair_index, **values, **support})
            gene_rows.extend({"pair_index": pair_index, "gene_argument_index": index, **row} for index, row in enumerate(gene_records))
            pd.DataFrame(pair_rows).to_csv(target / "pair_metrics.csv", index=False)
            save_json(OUTPUT_ROOT / "PROGRESS.json", {"dataset": dataset, "pairs_done": pair_index + 1, "pairs_expected": config["pairs"]})
            reference_slice = moving_slice
    assert len(pair_rows) == config["pairs"]
    means = {}
    for (display_name, native_key) in zip(METRICS, NATIVE_KEYS):
        numbers = [row[native_key] for row in pair_rows]
        assert len(numbers) == config["pairs"]
        means[display_name] = math.fsum(numbers) / len(numbers) if all(v is not None and math.isfinite(v) for v in numbers) else None
    pd.DataFrame(gene_rows).to_csv(target / "gene_metrics.csv.gz", index=False, compression="gzip")
    save_json(target / "RECOMPUTED.json", {"dataset": dataset, "pairs": config["pairs"], "metrics": means,
                                          "coordinates_sha256": config["coordinates_sha256"], "source_hashes_checked": VERIFY_SOURCE_HASHES})
    recomputed_rows.append({"Dataset": dataset, **means})
    print(f"{dataset}: {config['pairs']}/{config['pairs']} pairs recomputed")
recomputed = pd.DataFrame(recomputed_rows).set_index("Dataset")
recomputed.to_csv(OUTPUT_ROOT / "OmniAligner_NCS_recomputed.csv")
display(recomputed[METRICS].round(6))

DLPFC: 3/3 pairs recomputed


MHPR: 4/4 pairs recomputed


MMPC: 2/2 pairs recomputed


MPC: 2/2 pairs recomputed


E14_16: 15/15 pairs recomputed


MLN: 18/18 pairs recomputed


,PCC,CosSim,SSIM,MI,MSE,Region
Dataset,,,,,,
DLPFC,0.547930,0.783503,0.531970,0.299724,0.127579,0.884759
MHPR,0.540881,0.869558,0.527731,0.252730,1.218373,0.759039
MMPC,0.197483,0.666920,0.193019,0.069390,81.048677,0.871546
MPC,0.333290,0.625865,0.323609,0.152123,0.341752,0.978106
E14_16,0.411842,0.947845,0.414820,0.191671,0.125625,0.643624
MLN,0.812007,0.982588,0.811243,0.809955,0.055572,0.406042


## 5. Compare with the current result table
Each metric is checked separately at absolute tolerance `1e-10`, with no relative tolerance. A mismatch raises an error and leaves the recomputed files available for investigation; it never overwrites the reference table.

In [15]:
expected = reference.loc[DATASETS, METRICS].astype(float)
actual = recomputed.loc[DATASETS, METRICS].astype(float)
difference = actual - expected
matches = np.isclose(actual.to_numpy(), expected.to_numpy(), atol=ATOL, rtol=0, equal_nan=False)
display(difference.rename_axis("Dataset: recomputed minus reference"))
validation = {"all_metrics_match": bool(matches.all()), "datasets": DATASETS,
              "metrics_checked": int(matches.size), "absolute_tolerance": ATOL,
              "maximum_absolute_difference": float(np.abs(difference.to_numpy()).max()),
              "elapsed_seconds": time.time() - started,
              "source_hashes_checked": VERIFY_SOURCE_HASHES,
              "packages": {name: importlib.metadata.version(name) for name in ["numpy", "scipy", "pandas", "anndata", "scanpy", "scikit-learn"]}}
save_json(OUTPUT_ROOT / "VALIDATION.json", validation)
assert matches.all(), "Recomputed metrics differ: inspect pair_metrics.csv and gene_metrics.csv.gz; the reference was not changed."
print(f"PASS: {matches.size}/{matches.size} metrics match the reference at absolute tolerance {ATOL:g}.")

,PCC,CosSim,SSIM,MI,MSE,Region
Dataset: recomputed minus reference,,,,,,
DLPFC,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-2.775558e-17,0.000000e+00
MHPR,1.110223e-16,1.110223e-16,0.000000e+00,0.000000e+00,4.440892e-16,-1.110223e-16
MMPC,2.775558e-17,0.000000e+00,0.000000e+00,5.551115e-17,0.000000e+00,0.000000e+00
MPC,0.000000e+00,0.000000e+00,5.551115e-17,-5.551115e-17,0.000000e+00,0.000000e+00
E14_16,-5.551115e-17,-1.110223e-16,-5.551115e-17,2.775558e-17,0.000000e+00,0.000000e+00
MLN,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,6.245005e-17,5.551115e-17


PASS: 36/36 metrics match the reference at absolute tolerance 1e-10.


## Interpretation
Successful agreement verifies reproduction of these selected saved results under their original scoring protocol. It does not establish independent anatomical validation, blind model selection, or equivalence between annotation consistency and biological fidelity. The weights, coordinates, source inputs, gene panels and cohort definitions must travel together when reproducing the results.

## Reconstruct the complete comparison table
The OmniAligner rows below must use the newly computed pair scores. Other methods retain their disclosed numerical source and evaluation support.

## 1. Locate the frozen comparison sources
The OmniAligner pair scores were computed above. The other algorithms are reconstructed from the released pair-score records or archived means identified in the submission manifest.


In [16]:
from pathlib import Path
import os, json, csv, math, hashlib
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from IPython.display import display, Markdown, HTML
from html import escape

def locate_repository():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "notebooks/OmniAligner_NCS_manifest.json").is_file():
            return candidate
    raise FileNotFoundError("Run from this repository or its notebooks directory")
REPO_ROOT = locate_repository()
DATA_ROOT = REPO_ROOT / "data"
SAB_ROOT = REPO_ROOT
SOURCE_ROOT = DATA_ROOT / "sources"
OUTPUT_ROOT = REPO_ROOT / "results" / "table"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
METRICS = ["PCC", "CosSim", "SSIM", "MI", "MSE", "Region"]
NATIVE_KEYS = ["PCC", "CosSim", "SSIM", "MI", "Expression_MSE_native_ASSD_bygrid", "Accuracy"]
ATOL = 1e-10
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()
manifest = json.loads((REPO_ROOT / "notebooks/NCS_submission_manifest.json").read_text())
for filename, key in [("submission_scope.json", "scope_file_sha256"), ("archived_baselines.csv", "archived_baselines_sha256"),
                      ("submission_reference.csv", "reference_table_sha256"), ("submission_ranks_reference.csv", "reference_ranks_sha256")]:
    assert sha256(SOURCE_ROOT / filename) == manifest[key], filename
print("Submission candidate snapshot:", manifest["snapshot"])
print("Pending conditions:", manifest["pending"])
display(pd.DataFrame(manifest["bindings"])[["dataset", "method", "kind", "status"]])

Submission candidate snapshot: 2026-11-17T01:07:58-07:00
Pending conditions: []


,dataset,method,kind,status
0,DLPFC,OmniAligner,fresh_coordinate_recalculation,Scored
1,DLPFC,CAST,archived_scored_workbook,Scored
2,DLPFC,PASTE,archived_scored_workbook,Scored
3,DLPFC,PASTE2,archived_scored_workbook,Scored
4,DLPFC,PASTE (p0),archived_scored_workbook,Scored
5,DLPFC,SANTO,archived_scored_workbook,Scored
6,DLPFC,SLAT,archived_scored_workbook,Scored
7,DLPFC,SPACEL,archived_scored_workbook,Scored
8,DLPFC,STalign,archived_scored_workbook,Scored
9,DLPFC,MOSAICField (RNA),archived_scored_workbook,Scored


## 2. Reconstruct every numerical entry
Gene/pair definitions are inherited from the frozen original scoring records. Pair means require every expected pair and finite values for that metric. Insufficient-grid default molecular scores are displayed as NA; Region is retained separately. Historical workbook values are labelled as archived scored results rather than newly recomputed coordinates.

In [17]:
baseline_rows = {(r["Dataset"], r["Method"]): r for r in csv.DictReader((SOURCE_ROOT / "archived_baselines.csv").open())}
csv.field_size_limit(256 * 1024 * 1024)
def finite_or_none(value):
    try:
        value = float(value)
        return value if math.isfinite(value) else None
    except (ValueError, TypeError):
        return None
def complete_pair_means(binding):
    path = SAB_ROOT / binding["pair_metrics_path"]
    assert sha256(path) == binding["pair_metrics_sha256"], path
    if "summary_path" in binding:
        assert sha256(SAB_ROOT / binding["summary_path"]) == binding["summary_sha256"]
    rows = list(csv.DictReader(path.open()))
    if binding["kind"] == "fresh_coordinate_recalculation":
        fresh = REPO_ROOT / "results/omni" / binding["dataset"] / "pair_metrics.csv"
        assert fresh.is_file(), "Run the preceding coordinate-scoring cells first"
        if fresh.exists():
            fresh_rows = list(csv.DictReader(fresh.open()))
            assert len(fresh_rows) == len(rows)
            for saved, recalculated in zip(rows, fresh_rows):
                assert int(saved["pair_index"]) == int(recalculated["pair_index"])
                for key in NATIVE_KEYS:
                    assert np.isclose(float(saved[key]), float(recalculated[key]), atol=ATOL, rtol=0, equal_nan=True)
            rows = fresh_rows
    assert sorted(int(row["pair_index"]) for row in rows) == list(range(binding["pairs"]))
    fallback = any(row.get("native_insufficient_grid_fallback", "False") == "True" for row in rows)
    result = []
    for index, key in enumerate(NATIVE_KEYS):
        values = [finite_or_none(row[key]) for row in rows]
        value = math.fsum(values) / len(values) if all(v is not None for v in values) else None
        result.append(None if fallback and index < 5 else value)
    return result

rebuilt_rows = []
for binding in manifest["bindings"]:
    if binding["kind"] in ["fresh_coordinate_recalculation", "completed_pair_scores"]:
        values = complete_pair_means(binding)
    elif binding["kind"] == "archived_scored_workbook":
        values = [finite_or_none(baseline_rows[(binding["dataset"], binding["method"])][metric]) for metric in METRICS]
    else:
        assert binding["kind"] == "not_scored"
        values = [None] * 6
    rebuilt_rows.append({"Dataset": binding["dataset"], "Method": binding["method"], **dict(zip(METRICS, values)),
                         "Status": binding["status"], "Eligible": binding["eligible"]})
rebuilt = pd.DataFrame(rebuilt_rows)
assert not rebuilt["Method"].isin(["Loki", "CPD"]).any()
assert rebuilt[rebuilt.Method.eq("OmniAligner")].groupby("Dataset").size().eq(1).all()

def display_table(frame, cell_style, numeric_columns=()):
    parts = ['<table style="border-collapse:collapse;font:14px sans-serif"><thead><tr><th></th>']
    parts += [f'<th style="padding:8px">{escape(str(col))}</th>' for col in frame.columns]
    parts.append('</tr></thead><tbody>')
    for label, row in frame.iterrows():
        parts.append(f'<tr><th style="padding:8px;text-align:left">{escape(str(label))}</th>')
        for col in frame.columns:
            value = row[col]
            text = 'NA' if pd.isna(value) else f'{float(value):.6f}' if col in numeric_columns else str(value)
            parts.append(f'<td style="padding:8px;border:2px solid white;{cell_style(row, col)}">{escape(text)}</td>')
        parts.append('</tr>')
    parts.append('</tbody></table>')
    display(HTML(''.join(parts)))

for dataset, group in rebuilt.groupby("Dataset", sort=False):
    display(Markdown(f"### {dataset}"))
    table = group.set_index("Method")
    best = {metric: (table.loc[table.Eligible, metric].min() if metric == "MSE" else table.loc[table.Eligible, metric].max()) for metric in METRICS}
    def cell_style(row, col):
        return "background:#d2e8dc;color:#225b43;font-weight:bold" if row.Eligible and col in METRICS and pd.notna(row[col]) and row[col] == best[col] else "background:#f1f3f4;color:#243447"
    display_table(table[METRICS + ["Status", "Eligible"]], cell_style, METRICS)

### DLPFC

,PCC,CosSim,SSIM,MI,MSE,Region,Status,Eligible
OmniAligner,0.547930,0.783503,0.531970,0.299724,0.127579,0.884759,Scored,True
CAST,0.546627,0.781952,0.527965,0.313248,0.128917,0.869842,Scored,True
PASTE,0.544071,0.778515,0.528263,0.306357,0.125379,0.838851,Scored,True
PASTE2,0.543163,0.781307,0.528125,0.304160,0.127796,0.862013,Scored,True
PASTE (p0),0.540429,0.779052,0.526039,0.285680,0.128686,0.843411,Scored,True
SANTO,0.406315,0.723795,0.396180,0.177750,0.203190,0.496139,Scored,True
SLAT,0.536767,0.778471,0.521766,0.298667,0.133502,0.834437,Scored,True
SPACEL,0.542308,0.780266,0.525264,0.298687,0.120631,0.872916,Scored,True
STalign,0.531650,0.780250,0.515869,0.269830,0.144726,0.768149,Scored,True
MOSAICField (RNA),0.293491,0.681699,0.286240,0.128547,0.273680,0.425441,Scored,True


### MHPR

,PCC,CosSim,SSIM,MI,MSE,Region,Status,Eligible
OmniAligner,0.540881,0.869558,0.527731,0.252730,1.218373,0.759039,Scored,True
PASTE,0.014652,0.733618,0.026673,0.046316,4.015312,0.177458,Scored,True
PASTE (p0),0.504022,0.855469,0.492768,0.222687,1.538849,0.721494,Scored,True
SANTO,0.080521,0.197450,0.079817,0.026639,75.695452,0.136740,Scored,True
SLAT,0.510049,0.857475,0.497675,0.231305,1.490035,0.731335,Scored,True
SPACEL,0.528168,0.863517,0.515406,0.241713,1.307417,0.752253,Scored,True
STalign,0.000000,0.000000,0.000000,0.000000,100.000000,0.000000,Diagnostic; not ranked,False
Spateo nonrigid,0.535648,0.866232,0.521511,0.249683,1.281545,0.753327,Scored,True
Spateo rigid,0.536942,0.866585,0.522419,0.251475,1.267570,0.753625,Scored,True
PASTE2,0.521702,0.861287,0.509315,0.243140,1.393879,0.741250,Scored,True


### MMPC

,PCC,CosSim,SSIM,MI,MSE,Region,Status,Eligible
OmniAligner,0.197483,0.666920,0.193019,0.069390,81.048677,0.871546,Scored,True
PASTE,0.172212,0.650976,0.170476,0.067978,81.590440,0.828582,Scored,True
SANTO,0.099983,0.606989,0.104715,0.045556,93.775101,0.721195,Scored,True
SLAT,0.183108,0.655952,0.181560,0.064057,79.690503,0.831573,Scored,True
SPACEL,0.188059,0.655706,0.183758,0.062148,83.686211,0.902289,Scored,True
STalign,0.185367,0.659121,0.183038,0.068431,73.721764,0.801403,Scored,True
PASTE2,NA,NA,NA,NA,NA,NA,Failed: 0/2 pairs,False
CAST,0.187550,0.645275,0.185057,0.061726,78.848240,0.894540,New: complete and scored,True
MOSAICField (RNA),0.128647,0.591948,0.129254,0.049071,101.993291,0.621480,New: complete and scored,True


... (2 outputs left out)

### E14_16

,PCC,CosSim,SSIM,MI,MSE,Region,Status,Eligible
OmniAligner,0.411842,0.947845,0.414820,0.191671,0.125625,0.643624,Scored,True
PASTE,0.382526,0.944291,0.386579,0.162621,0.136930,0.563724,Scored,True
SLAT,0.389036,0.944710,0.392797,0.165479,0.135331,0.579770,Scored,True
SPACEL,0.409362,0.945594,0.411990,0.192494,0.131897,0.635016,Scored,True
PASTE2,0.384870,0.939064,0.386243,0.171316,0.149774,0.592428,Scored,True
CAST,NA,NA,NA,NA,NA,0.126806,Native fallback; Region scored,True
MOSAICField (RNA),0.182823,0.914408,0.194066,0.089574,0.211897,0.372768,New: complete and scored,True
SANTO,0.321615,0.935610,0.327376,0.128812,0.157290,0.470798,New: complete and scored,True
STalign,0.385493,0.944777,0.389318,0.166618,0.134542,0.576439,New: complete and scored,True


### MLN

,PCC,CosSim,SSIM,MI,MSE,Region,Status,Eligible
OmniAligner,0.812007,0.982588,0.811243,0.809955,0.055572,0.406042,Scored,True
SLAT full sample,0.628273,0.955256,0.630293,0.410136,0.085370,0.263900,Scored,True
SPACEL,0.810401,0.981903,0.807871,0.797399,0.056288,0.406349,Scored,True
STalign,0.007397,0.148610,0.010834,0.010396,83.371461,0.032328,Scored,True
Spateo nonrigid,0.813540,0.980730,0.810787,0.734435,0.055805,0.378206,Scored,True
PASTE (5k),0.787607,0.979748,0.784433,0.586253,0.056072,0.429433,Scored,True
PASTE2 (5k),0.668936,0.961677,0.667826,0.430011,0.077474,0.306138,New: complete and scored,True
MOSAICField (RNA),0.671677,0.960303,0.671442,0.464766,0.082297,0.285989,New: complete and scored,True
SANTO,0.531558,0.939827,0.528887,0.306617,0.108158,0.236073,New: complete and scored,True
CAST,NA,NA,NA,NA,NA,NA,Incomplete; not ranked,False


## 3. Recompute algorithm ranks and denominators
Count each algorithm once, using its best eligible configuration for each metric. PASTE configurations count as PASTE, Spateo configurations as Spateo; PASTE2 remains separate. Undefined, failed, incomplete and diagnostic values do not enter that metric's denominator. Rank 1 is green; other ranks are gray.

In [18]:
def algorithm(name):
    for prefix in ["OmniAligner", "PASTE2", "PASTE", "Spateo", "SLAT", "MOSAICField"]:
        if name.startswith(prefix):
            return prefix
    return name
rank_rows = []
for dataset, group in rebuilt.groupby("Dataset", sort=False):
    for metric in METRICS:
        valid = group.loc[group.Eligible & group[metric].notna()].copy()
        valid["Algorithm"] = valid.Method.map(algorithm)
        scores = valid.groupby("Algorithm")[metric].agg("min" if metric == "MSE" else "max")
        value = scores["OmniAligner"]
        rank = 1 + int((scores < value).sum() if metric == "MSE" else (scores > value).sum())
        best = scores.min() if metric == "MSE" else scores.max()
        rank_rows.append({"Dataset": dataset, "Metric": metric, "OmniAligner score": value, "Rank": rank,
                          "Algorithms": len(scores), "Displayed rank": f"{rank}/{len(scores)}",
                          "Best algorithm(s)": "; ".join(sorted(scores.index[scores.eq(best)]))})
ranks = pd.DataFrame(rank_rows)
matrix = ranks.pivot(index="Dataset", columns="Metric", values="Displayed rank").reindex(index=rebuilt.Dataset.drop_duplicates(), columns=METRICS)
display_table(matrix, lambda row, col: "background:#d2e8dc;color:#225b43;font-weight:bold" if row[col].startswith("1/") else "background:#f1f3f4;color:#243447")
print("OmniAligner rank-one cells:", int(ranks.Rank.eq(1).sum()), "/", len(ranks))

,PCC,CosSim,SSIM,MI,MSE,Region
DLPFC,1/9,1/9,1/9,4/9,3/9,1/9
MHPR,1/9,1/9,1/9,1/9,1/9,2/9
MMPC,1/8,1/8,1/8,1/8,4/8,3/8
MPC,2/5,2/5,2/5,1/9,3/9,1/9
E14_16,1/8,1/8,1/8,2/8,1/8,1/9
MLN,2/9,1/9,1/9,1/9,1/9,3/9


OmniAligner rank-one cells: 24 / 36


## 4. Verify the entire submission table and ranking
The comparison checks all dataset–method rows, every numeric value and NA position, eligibility, status, ranks and denominators. Outputs are written to a separate directory; the reference table is not modified.

In [19]:
expected = pd.read_csv(SOURCE_ROOT / "submission_reference.csv")
assert rebuilt[["Dataset", "Method"]].values.tolist() == expected[["Dataset", "Method"]].values.tolist()
assert rebuilt.Eligible.tolist() == expected.Eligible.tolist()
assert rebuilt.Status.tolist() == expected.Status.tolist()
actual_values = rebuilt[METRICS].to_numpy(dtype=float)
expected_values = expected[METRICS].to_numpy(dtype=float)
assert np.array_equal(np.isnan(actual_values), np.isnan(expected_values)), "NA pattern differs"
assert np.allclose(actual_values, expected_values, rtol=0, atol=ATOL, equal_nan=True), "A table value differs"
expected_ranks = pd.read_csv(SOURCE_ROOT / "submission_ranks_reference.csv")
for column in ["Dataset", "Metric", "Rank", "Algorithms", "Displayed rank"]:
    assert ranks[column].tolist() == expected_ranks[column].tolist(), column
for actual, saved in zip(ranks["Best algorithm(s)"], expected_ranks["Best algorithm(s)"]):
    assert set(actual.split("; ")) == set(saved.split("; "))
rebuilt.to_csv(OUTPUT_ROOT / "NCS_submission_table_reproduced.csv", index=False)
ranks.to_csv(OUTPUT_ROOT / "NCS_submission_ranks_reproduced.csv", index=False)
report = {"all_table_entries_match": True, "all_rank_cells_match": True, "method_rows": len(rebuilt),
          "metric_positions": int(actual_values.size), "finite_metric_positions": int(np.isfinite(actual_values).sum()),
          "rank_cells": len(ranks), "absolute_tolerance": ATOL,
          "maximum_absolute_difference": float(np.nanmax(np.abs(actual_values - expected_values))),
          "pending": manifest["pending"], "snapshot": manifest["snapshot"],
          "comparison_scope": "entire submission candidate table and rank matrix",
          "evidence_modes": pd.Series([b["kind"] for b in manifest["bindings"]]).value_counts().to_dict()}
(OUTPUT_ROOT / "VALIDATION.json").write_text(json.dumps(report, indent=2))
display(pd.DataFrame([report]).drop(columns=["pending", "evidence_modes"]))
print("PASS: complete submission table and ranking reproduced.")
if manifest["pending"]:
    print("Pending rows remain unscored. Freeze the updated reference after completion; this snapshot is not a claim that those runs have finished.")

,all_table_entries_match,all_rank_cells_match,method_rows,metric_positions,finite_metric_positions,rank_cells,absolute_tolerance,maximum_absolute_difference,snapshot,comparison_scope
0,True,True,59,354,325,36,1.000000e-10,4.440892e-16,2026-11-17T01:07:58-07:00,entire submission candidate table and rank matrix


PASS: complete submission table and ranking reproduced.
